<a href="https://colab.research.google.com/github/lestermartin/starburst-dataframes-exploration/blob/main/LazyExecution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing eager & lazy execution models with dataframe operations

This module will compare & contrast the run-time characteristics of these different dataframe implementation approaches.

*   **Eager execution** using [Pandas](https://pandas.pydata.org/)
*   **Lazy execution** using [Ibis](https://ibis-project.org/)

![garfield](https://somospnt.com/images/blog/cover/243-fetchtype-con-jpa.jpg)

# But first, some review...

## What is a dataframe?

![dataframe](https://pandas.pydata.org/docs/_images/01_table_dataframe.svg)

Dataframes are two-dimensional, tabular data structures with labeled & typed columns where each column can be a different data type (numbers, strings, dates, etc), but values within a column are typically the same type.

Conceptually, think of a variable holding some, or all, of the rows from a database table.

### Example dataframe

The following table visualizes a dataframe consisting of rows describing penguins that are part of a research project.

| species | island | bill_length_mm | bill_depth_mm | flipper_length_mm | body_mass_g | sex | year |
|---|---|---|---|---|---|---|---|
| Adelie | Torgersen | 39.1 | 18.7 | 181 | 3750 | male | 2007 |
| Adelie | Torgersen | 39.5 | 17.4 | 186 | 3800 | female | 2007 |
| Adelie | Biscoe | 40.6 | 18.6 | 183 | 3550 | male | 2008 |
| Adelie | Dream | 36.6 | 17.8 | 185 | 3700 | female | 2008 |
| Gentoo | Biscoe | 46.1 | 13.2 | 211 | 4500 | female | 2009 |
| Gentoo | Biscoe | 50.0 | 16.3 | 230 | 5700 | male | 2009 |
| Chinstrap | Dream | 46.5 | 17.9 | 192 | 3500 | female | 2008 |
| Chinstrap | Dream | 50.0 | 19.5 | 196 | 3900 | male | 2008 |
| Gentoo | Biscoe | 45.2 | 14.8 | 212 | 5200 | female | 2007 |
| Adelie | Dream | 37.3 | 16.8 | 182 | 3400 | female | 2009 |



## Dataframe APIs

![api](https://www.cleo.com/sites/default/files/api-integration.png)

Many of the functions available to dataframe objects are referred to as transformation functions. This means they return a new dataframe object or alter the dataframe is some way.

The nuance of that last statement hinges on the topic we are looking at the run-time characteristics of; **eager vs. lazy execution**.

### Eager execution

This is the typical model most learn about when they start to program. As a program executes line-by-line and it encounters a dataframe function that retrieves or modifies data in some way, it happens immediately.

*  The dataframe's contents live in memory and are accessible by the single process space
*  The dataframe is mutable and transformation functions can make needed modifications to the dataframe's contents


### Lazy execution

As you move into distributed compute engines, such as [Apache Spark](https://spark.apache.org/) and Trino, **the actual I/O required to retrieve or modify data is deferred as long as possible**.

*  Each dataframe object is actually a *set of instructions* of how to get and modify data when it actually begins the execution of the I/O activities *--they do **NOT** contain the actual data*
*  Dataframes are immutable so any modification requires the creation of a new dataframe
*  Work is actually triggered when I/O **needs** to occur; such as displaying or persisting final results
*  The processing itself is executed on, and coordinated across, multiple nodes within a cluster

**It is critical that you understand this notion of lazy execution for this module. Please revisit the prerequisite module if this refresher is not clear.**

For those that "get it", but want a deeper dive into what's going on in these distributed engines, check out the materials presented in this [video series](https://lestermartin.blog/2025/04/22/trino-query-plan-analysis-video-series/).

![query plan](https://i0.wp.com/lestermartin.blog/wp-content/uploads/2025/04/TrinoQueryPlan2.png?resize=768%2C320&ssl=1)


## Show me some code again...

Remember, we've already explored both of the frameworks and their APIs, but here is a quick comparison accessing the penguins data shown earlier. Both examples

![pandas](https://pandas.pydata.org/docs/_static/pandas.svg)

```python
import pandas as pd

df = pd.read_csv("penguins.csv")

result = (
    df.groupby(["species", "island"])
      .size()
      .reset_index(name="count")
      .sort_values("count")
)
```

![ibis](https://ibis-project.org/logo.svg)

```python
import ibis

t = ibis.read_csv("penguins.csv")

result = (
    t.group_by(["species", "island"])
     .agg(count=ibis._.count())
     .order_by("count")
)
```


# Env setup

The hands-on examples in this notebook access data via [Trino](https://trino.io/) and its [TPC-H connector](https://trino.io/docs/current/connector/tpch.html). If needed, here are two simple ways to set up such an environment.

*  Run [Trino in a Docker container](https://trino.io/docs/current/installation/containers.html) on your workstation
*  Register for a [Starburst Galaxy](https://www.starburst.io/starburst-galaxy/) hosted environment


## Select TPC-H form factor

For initial testing, enter `tiny`, but scale up to `sf1`, `sf5`, `sf10`, etc as needed.

In [ ]:
import getpass

# select form factory
tpch_form_factor = input("Form factor (defaults to 'tiny')"
)
if tpch_form_factor == "":
  tpch_form_factor = "tiny"

print("\nForm factor set to", tpch_form_factor)

## Input your Trino connection details

In [ ]:
# grab credentials from the notebook user to be used when making a connection
my_host = input("Host name")
my_username = input("User name")
my_password = getpass.getpass("Password")

## Setup and verify [Trino Python client](https://github.com/trinodb/trino-python-client)

In [ ]:
# install Trino Python client

%pip install trino

In [ ]:
# boiler-plate code for setup

from trino.dbapi import connect
from trino.auth import BasicAuthentication

# sanity check
print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')


# build the connection object with the hostname & creds inputed earlier
conn = connect(
    host=my_host,
    port="443",
    user=my_username,
    auth=BasicAuthentication(my_username, my_password),
    http_scheme="https",
    catalog="system",
    schema="runtime",
)
cur = conn.cursor()
cur.execute("SELECT '** CONNECTION IS GOOD **'")
rows = cur.fetchall()
print(rows)

## Setup and verify [Ibis](https://ibis-project.org/backends/trino) for Trino backend

NOTE: Ibis can run against many different SQL engines, not just Trino.

In [ ]:
# install Ibis

%pip install 'ibis-framework[trino]'

In [ ]:
# boiler-plate code for setup

import os
import ibis
from trino.auth import BasicAuthentication

ibis.options.interactive = True

user = my_username
trino_auth_obj = BasicAuthentication(my_username, my_password)
host = my_host
port = "443"
http_scheme = "https"
catalog = "tpch"
schema = "tiny"

con = ibis.trino.connect(
    user=user, auth=trino_auth_obj, host=host, port=port, http_scheme=http_scheme, database=catalog, schema=schema
)

print('\n Make sure the phrase ** CONNECTION IS GOOD ** displays \n')
con.sql("select '** CONNECTION IS GOOD **' as conn_check")

## Use case

You have a schema which includes these 3 tables.

```
┌────────────────────────┐
│        NATION          │
├────────────────────────┤
│ PK  nationkey          │
│     name               │
│     ...addt cols...    │
└────────────────────────┘
            ▲
            │ 1
            │
            │ N
┌────────────────────────┐
│        CUSTOMER        │
├────────────────────────┤
│ PK  custkey            │
│     acctbal            │
│ FK  nationkey          │
│     ...addt cols...    │
└────────────────────────┘
            ▲
            │ 1
            │
            │ N
┌────────────────────────┐
│         ORDERS         │
├────────────────────────┤
│ PK  orderkey           │
│ FK  custkey            │
│     orderstatus        │
│     orderpriority      │
│     totalprice         │
│     ...addt cols...    │
└────────────────────────┘
```

For all customers with an account balance > $9900, find the total number of open orders and their average price at the unique intersection of country and order priority values; ordering them by that same intersection point.

## do work

In [ ]:
import time
import pandas as pd

# Create the hashmap (dictionary) with operation_name -> operation_time
pandas_ops = {}

op_name = "create customer DF"

# create a pandas DF with customer data
start_time = time.time() #noise

# -------------------------
cur.execute("SELECT * FROM tpch." + tpch_form_factor + ".customer")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_cust_df = pd.DataFrame(rows, columns=col_name)
# -------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise


# display a few customers
p_init_cust_df.head(3)

In [ ]:
op_name = "keep large balance customers"

# filter out customers with smaller balances
start_time = time.time() #noise

# -------------------------
p_lrg_bal_cust_df = p_init_cust_df[p_init_cust_df['acctbal'] > 9900.0]
# -------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise

p_lrg_bal_cust_df.head(3)

In [ ]:
p_limited_cols_cust_df = p_lrg_bal_cust_df[['custkey','nationkey']]

p_best_cust_df = p_limited_cols_cust_df.rename(columns={'custkey': 'c_custkey', 'nationkey': 'c_nationkey'})

p_best_cust_df.head(3)


In [ ]:
cur.execute("SELECT * FROM tpch." + tpch_form_factor + ".nation")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_nation_df = pd.DataFrame(rows, columns=col_name)


p_init_nation_df.head(3)


In [ ]:
p_limited_cols_nation_df = p_init_nation_df.drop(columns=['regionkey', 'comment'])

p_best_nation_df = p_limited_cols_nation_df.rename(columns={'name': 'n_name', 'nationkey': 'n_nationkey'})

p_best_nation_df.head(3)

In [ ]:
p_init_join_c_n_df = p_best_cust_df.merge(p_best_nation_df, left_on='c_nationkey', right_on='n_nationkey')

#p_sorted_c_n_df = p_init_join_c_n_df.sort_values(by='acctbal', ascending=False)

p_best_join_c_n_df = p_init_join_c_n_df.drop(columns=['c_nationkey', 'n_nationkey'])

p_best_join_c_n_df.head(3)

In [ ]:
op_name = "create orders DF"

# create a pandas DF with orders data
start_time = time.time() #noise

# -------------------------
cur.execute("SELECT * FROM tpch." + tpch_form_factor + ".orders")
rows = cur.fetchall()
col_name = [desc[0] for desc in cur.description]
p_init_orders_df = pd.DataFrame(rows, columns=col_name)
# -------------------------

op_time = time.time() - start_time #noise
pandas_ops[op_name] = op_time #noise
print(f"{op_name} took {op_time} secs") #noise


# display a few customers
p_init_orders_df.head(3)

In [ ]:
p_limited_cols_orders_df = p_init_orders_df[['custkey', 'orderstatus', 'orderpriority', 'totalprice']]

p_rn_lim_cols_orders_df = p_limited_cols_orders_df.rename(columns={'custkey': 'o_custkey'})

p_rn_lim_cols_orders_df.head(3)

In [ ]:
p_best_orders_df = p_rn_lim_cols_orders_df[p_rn_lim_cols_orders_df['orderstatus'] == 'O']

p_best_orders_df.head(3)


In [ ]:
p_init_join_o_cn_df = p_best_orders_df.merge(p_best_join_c_n_df, left_on='o_custkey', right_on='c_custkey')

p_best_full_join_df = p_init_join_o_cn_df.drop(columns=['o_custkey'])

p_best_full_join_df.head(3)

In [ ]:
result = (
    p_best_full_join_df.groupby(["n_name", "orderpriority"])
    .agg(
        total_count=("c_custkey", "count"),
        avg_price=("totalprice", "mean")
    )
    .reset_index()
    .sort_values(["n_name", "orderpriority"])
)

result


In [ ]:
# --- List the entire hashmap ---
print("All Pandas operations:")
for operation_name, operation_time in pandas_ops.items():
    print(f"  {operation_name}: {operation_time}s")

# --- Total up the operation_time values ---
total_time = sum(pandas_ops.values())
print(f"\nTotal Pandas operation time: {total_time}s")